**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/pakhomov-2023"], check=True)
    os.chdir("/content/encore/entries/pakhomov-2023")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Earlier findings of pakhomov-2023-04, checked on the current versions

This notebook belongs to the reconciliation of attempt `pakhomov-2023-05` with the earlier attempt
`pakhomov-2023-04` (docs/protocol.md, "Repeat attempts"). The earlier attempt tested dandiset
[DANDI:001027](https://dandiarchive.org/dandiset/001027/0.240605.1859) at version 0.240605.1859, the
version pinned here (`cancan.VERSIONS`). It recorded four findings. Findings 1 and 2 (the data of
Fig. 4 not deposited; the Protocol 2 and Protocol 3 labels) are reported again by `pakhomov-2023-05`'s
own findings 1 and 5, and finding 3 in part by its finding 2. This notebook checks what those do not
cover: the part of finding 3 on exposure, gain and background (section 1) and finding 4 (section 2).

The files are streamed from the archive at the pinned version, one at a time, and each image is
reduced to block means of 8 x 8 pixels as in `cancan.load_image`.

In [ ]:
import io
import re

import h5py
import numpy as np
import pandas as pd
import requests

import cancan as cc

pd.set_option("display.width", 200)
DS = "001027"
API = f"{cc.API}/dandisets/{DS}/versions/{cc.VERSIONS[DS]}"


def attr_text(v):
    if isinstance(v, bytes):
        return v.decode()
    if isinstance(v, np.ndarray) and v.dtype.kind in "SO":
        return " ".join(x.decode() if isinstance(x, bytes) else str(x) for x in v.ravel())
    return str(v)


def read_dish(asset):
    """Metadata of one file, every field naming exposure or gain, and its reduced image."""
    meta = cc._asset_meta(DS, asset["asset_id"])
    url = [u for u in meta["contentUrl"] if "s3" in u][0]
    buf = cc._fetch(url, meta["contentSize"])
    hits = []
    with h5py.File(io.BytesIO(buf), "r") as f:
        def visit(name, obj):
            if name.startswith("specifications"):
                return  # the NWB schema text, not this file's metadata
            texts = [name] + [f"{k}={attr_text(v)}" for k, v in obj.attrs.items()]
            if isinstance(obj, h5py.Dataset) and obj.dtype.kind in "SO" and obj.size == 1:
                texts.append(attr_text(obj[()]))
            hits.extend(t[:120] for t in texts if re.search(r"exposure|gain", t, re.I))
        f.visititems(visit)
        ims = f["acquisition/ScanningImaging"]
        d = ims["data"]
        rec = dict(
            subject=f["general/subject/subject_id"][()].decode(),
            start=f["session_start_time"][()].decode(),
            species=f["general/subject/species"][()].decode(),
            age=f["general/subject/age"][()].decode(),
            resolution=float(d.attrs["resolution"]), unit=attr_text(d.attrs["unit"]),
            image_description=attr_text(ims.attrs.get("description", "")),
            devices=list(f["general/devices"]) if "general/devices" in f else [],
            description=f["session_description"][()].decode(),
            exposure_or_gain_fields=hits,
        )
        im = d[0]
        h, w = im.shape[0] // cc.K * cc.K, im.shape[1] // cc.K * cc.K
        rec["frame"] = im[:h, :w].reshape(h // cc.K, cc.K, w // cc.K, cc.K).mean((1, 3), dtype=np.float32)
        del im
    del buf
    m = re.search(r"Protocol(\d)", rec["description"])
    rec["packet"] = int(m.group(1)) if m else None
    return rec


summary = requests.get(API + "/", timeout=60).json()["assetsSummary"]
dishes = [read_dish(a) for a in cc.list_assets(DS)]
print(len(dishes), "files of dandiset", DS, "version", cc.VERSIONS[DS])

In [ ]:
tab = pd.DataFrame([{k: d[k] for k in ("subject", "packet", "start", "resolution", "unit", "age", "species", "devices")}
                    for d in dishes]).sort_values(["packet", "start"])
print(tab.to_string(index=False))
print("\nimage shapes (reduced 8 x 8):", sorted({d["frame"].shape for d in dishes}))
print("reduced intensity range:", round(float(min(d["frame"].min() for d in dishes)), 1), "to",
      round(float(max(d["frame"].max() for d in dishes)), 1), "a.u. (unit as recorded:", sorted(set(tab.unit)), ")")

## 1. Earlier finding 3, exposure, gain and background

The earlier statement: "The images have no pixel size (ImageSeries resolution is 96 in nine files and
1.0 in one, unit "n.a."), so distances, which the paper's ROIs are defined by (+/-1.65 mm), rest on
the camera's pixel and the objective. The files do not record where the electrodes are or which is
e1, so they were found in the images and a reading that needs e1 could not be run. Exposure and gain
are not recorded; the three Packet 1 dishes, imaged on 24 March 2021, have about twice the background
of the seven others (mean 308 a.u., against 162 and 150 on 17 and 18 March), so a.u. values may not be
comparable between imaging days."

The pixel size and the electrodes are reported by finding 2 of `pakhomov-2023-05`. Here: the
`resolution` of each file, any field of any file that names exposure or gain, and the background as
the earlier attempt measured it, the median of the reduced image inside the central 60% of the radius
of each electrode footprint (cells under the needle, not exposed), by imaging date. The footprints
are found with `cancan.find_footprints`.

In [ ]:
print("resolution values:", tab.resolution.value_counts().to_dict())
print("fields naming exposure or gain, over all files:", sum(len(d["exposure_or_gain_fields"]) for d in dishes))
for d in dishes:
    for t in d["exposure_or_gain_fields"]:
        print("  ", d["subject"], t)

rows = []
for d in dishes:
    g = cc.find_footprints(d["frame"], "small")
    yy, xx = np.mgrid[:d["frame"].shape[0], :d["frame"].shape[1]]
    each = []
    for x, y in g["P"]:
        inside = (xx - x) ** 2 + (yy - y) ** 2 <= (0.6 * g["r"]) ** 2
        v = d["frame"][inside & (d["frame"] > 0)]
        each.append(float(np.median(v)))
    rows.append(dict(subject=d["subject"], packet=d["packet"], date=d["start"][:10],
                     fit_error=round(g["err"], 3), background=round(float(np.mean(each)), 1),
                     per_footprint=[round(v, 1) for v in each]))
bg = pd.DataFrame(rows).sort_values(["packet", "subject"])
print("\nmedian inside each footprint (central 60% of the radius), a.u.:")
print(bg.to_string(index=False))
print("\nby imaging date:")
print(bg.groupby("date").background.agg(["count", "mean", "min", "max"]).round(1).to_string())
by_date = bg.groupby("date").background.mean()
print("\nratio of the 24 March mean to the mean of the other dishes:",
      round(float(by_date.get("2021-03-24") / bg[bg.date != "2021-03-24"].background.mean()), 2))

**Result:** still present. The `resolution` field is 96 in nine files and 1.0 in dish 10, with unit "n.a.", so no file gives a pixel size, and no field of any file names an exposure or a gain. The background inside the footprints is 308.9 a.u. on average in the three Packet 1 dishes imaged on 24 March 2021, against 161.1 (17 March, six dishes) and 150.0 (18 March, dish 10), 1.94 times the other dishes, as the earlier attempt found (308, 162 and 150 a.u.).

## 2. Earlier finding 4, species, age and the record of dish 10

The earlier statement: "The dandiset's species summary is "Cricetulus griseus - Cricetulus aureus"
for one cell line (CHO, Chinese hamster); each subject is a dish with age P1D, which does not apply
to a cell line. The file of dish 10 names no microscope, camera or objective, where the other nine
name the ORCA-Flash4.0 and a 10x objective, and its session starts on 18 March 2021 at 18:50, the
evening after the other Packet 2 and 3 dishes."

Here: the species in the dandiset's assets summary and in each file, the subject age, whether each
ImageSeries description names the camera and objective, and the session starts.

In [ ]:
print("assets summary species:", [(s["name"], s["identifier"]) for s in summary["species"]])
print("species in the files:", sorted(set(tab.species)))
print("ages in the files:", sorted(set(tab.age)))
dev = pd.DataFrame([dict(subject=d["subject"], packet=d["packet"], start=d["start"][:16],
                         names_orca="ORCA" in d["image_description"],
                         names_10x=bool(re.search(r"Mag=10\b|10x", d["image_description"])),
                         description_chars=len(d["image_description"]))
                    for d in dishes]).sort_values(["packet", "start"])
print(dev.to_string(index=False))
for d in dishes:
    if "ORCA" not in d["image_description"]:
        print("\nImageSeries description of dish", d["subject"], ":", d["image_description"][:600])
print("\none ImageSeries description naming the camera:",
      next(d["image_description"] for d in dishes if "ORCA" in d["image_description"])[:600])

**Result:** still present. The dandiset's assets summary names the species "Cricetulus griseus - Cricetulus aureus" (NCBITaxon 10029); every file gives NCBITaxon 10029 and age P1D. Nine ImageSeries descriptions name the Hamamatsu ORCA-Flash4.0 and an objective of Mag=10; that of dish 10 gives "Unknown" microscope, detector and objective, and its session starts on 18 March 2021 at 18:50, where the other Packet 2 and 3 dishes start on 17 March between 11:36 and 13:03.